# STEMGym Benchmark Analysis

This notebook generates paper figures from benchmark results:
1. **DEC Curves** — Dose-Efficiency Curves (main result)
2. **Ablation Bar Chart** — DEC-AUC with each component removed
3. **Decision Trace** — Agent trajectory + planner mode transitions
4. **Sim vs Replay Correlation** — Agent rankings across data sources
5. **Summary Table** — Full results table for the paper

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib
import numpy as np

# Style for paper figures
matplotlib.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "legend.fontsize": 11,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

sys.path.insert(0, str(Path(".").resolve().parent))

RESULTS_DIR = Path(".").resolve().parent / "results"
FIGURES_DIR = Path(".").resolve().parent / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

# Agent display names and colors for consistent plotting
AGENT_STYLE = {
    "random":         {"label": "Random",          "color": "#aaaaaa", "ls": ":",  "marker": "x"},
    "raster":         {"label": "Raster",          "color": "#66c2a5", "ls": "--", "marker": "s"},
    "gp_ucb":         {"label": "GP-UCB",          "color": "#fc8d62", "ls": "-.", "marker": "^"},
    "single_bo":      {"label": "Single-BO",       "color": "#8da0cb", "ls": "--", "marker": "v"},
    "stem_agent":     {"label": "STEMAgent (Full)", "color": "#e78ac3", "ls": "-",  "marker": "o"},
    "no_planner":     {"label": "No Planner",      "color": "#a6d854", "ls": "--", "marker": "D"},
    "no_uncertainty": {"label": "No Uncertainty",   "color": "#ffd92f", "ls": "--", "marker": "p"},
    "rule_planner":   {"label": "Rule Planner",    "color": "#e5c494", "ls": "-.", "marker": "h"},
    "llm_planner":    {"label": "LLM Planner",     "color": "#b3b3b3", "ls": "-",  "marker": "*"},
}

TASK_LABELS = {
    "defect_census": "Defect Census (F1)",
    "phase_mapping": "Phase Mapping (IoU)",
    "targeted":      "Targeted (F1)",
}


def load_results(name: str) -> dict | None:
    path = RESULTS_DIR / f"{name}.json"
    if not path.exists():
        print(f"Warning: {path} not found. Run the benchmark first.")
        return None
    with open(path) as f:
        return json.load(f)


def get_style(agent_name: str) -> dict:
    return AGENT_STYLE.get(agent_name, {"label": agent_name, "color": "gray", "ls": "-", "marker": "."})


print("Setup complete.")

## 1. DEC Curves (Main Result)

Plot Dose-Efficiency Curves showing task score vs. fraction of dose budget used.
Each agent's curve shows how efficiently it uses its electron dose budget.

In [ ]:
def plot_dec_curves_from_episodes(
    world_path: str,
    task_name: str,
    agent_names: list[str],
    budget: float = 5000.0,
    max_steps: int = 200,
    seeds: int = 3,
    score_every: int = 3,
    ax=None,
):
    """Run agents and plot DEC curves directly (for when you don't have pre-saved results)."""
    from stem_gym.envs.stem_env import STEMEnv
    from stem_gym.runner import run_benchmark
    from experiments.run_benchmark import make_agent, make_task

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))

    task = make_task(task_name)

    for agent_name in agent_names:
        env = STEMEnv(world_path, task=task, total_budget=budget, max_steps=max_steps)
        agent = make_agent(agent_name)
        bench = run_benchmark(env, agent, agent_name, n_seeds=seeds,
                              score_every_n_acquires=score_every)
        env.close()

        style = get_style(agent_name)

        # Average DEC curves across seeds
        all_fracs = bench.episode_results[0].dec.dose_fractions
        all_scores = np.array([r.dec.scores for r in bench.episode_results])
        mean_scores = all_scores.mean(axis=0)
        std_scores = all_scores.std(axis=0)

        ax.plot(all_fracs, mean_scores,
                label=f"{style['label']} (AUC={bench.mean_dec_auc:.3f})",
                color=style["color"], ls=style["ls"], lw=2)
        ax.fill_between(all_fracs, mean_scores - std_scores, mean_scores + std_scores,
                        alpha=0.15, color=style["color"])

    ax.set_xlabel("Dose Fraction Used")
    ax.set_ylabel(f"Task Score ({TASK_LABELS.get(task_name, task_name)})")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)
    return ax


# Plot DEC for each task on the test world
world = str(Path("..") / "data" / "simulated" / "test_world.h5")

agents = ["random", "raster", "gp_ucb", "stem_agent"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, task_name in enumerate(["defect_census", "phase_mapping", "targeted"]):
    try:
        plot_dec_curves_from_episodes(
            world, task_name, agents,
            budget=5000.0, seeds=3, score_every=3,
            ax=axes[i],
        )
        axes[i].set_title(TASK_LABELS.get(task_name, task_name))
    except Exception as e:
        axes[i].text(0.5, 0.5, f"Error: {e}", transform=axes[i].transAxes,
                     ha="center", va="center", fontsize=10)
        axes[i].set_title(task_name)

fig.suptitle("Dose-Efficiency Curves (DEC) — Test World", fontsize=16, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "dec_curves.pdf")
fig.savefig(FIGURES_DIR / "dec_curves.png")
plt.show()
print(f"Saved to {FIGURES_DIR / 'dec_curves.pdf'}")

## 2. Ablation Bar Chart

Bar chart comparing DEC-AUC across ablation variants.
Shows contribution of each component (planner, uncertainty).

In [ ]:
def plot_ablation_bar(
    world_path: str,
    task_name: str = "defect_census",
    budget: float = 5000.0,
    seeds: int = 3,
    ax=None,
):
    """Run ablation variants and plot bar chart of DEC-AUC."""
    from stem_gym.envs.stem_env import STEMEnv
    from stem_gym.runner import run_benchmark
    from experiments.run_benchmark import make_agent, make_task

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))

    ablation_agents = ["stem_agent", "no_planner", "no_uncertainty", "rule_planner"]
    task = make_task(task_name)

    names = []
    aucs = []
    stds = []
    colors = []

    for agent_name in ablation_agents:
        env = STEMEnv(world_path, task=task, total_budget=budget, max_steps=200)
        agent = make_agent(agent_name)
        bench = run_benchmark(env, agent, agent_name, n_seeds=seeds, score_every_n_acquires=3)
        env.close()

        style = get_style(agent_name)
        names.append(style["label"])
        aucs.append(bench.mean_dec_auc)
        stds.append(bench.std_dec_auc)
        colors.append(style["color"])

    x = np.arange(len(names))
    bars = ax.bar(x, aucs, yerr=stds, capsize=5, color=colors, edgecolor="black", alpha=0.85)

    # Add value labels on bars
    for bar, auc in zip(bars, aucs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{auc:.3f}", ha="center", va="bottom", fontsize=10)

    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=15, ha="right")
    ax.set_ylabel("DEC-AUC")
    ax.set_title(f"Ablation Study — {TASK_LABELS.get(task_name, task_name)}")
    ax.set_ylim(0, max(aucs) * 1.3 if aucs else 1)
    ax.grid(axis="y", alpha=0.3)
    return ax


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, task_name in enumerate(["defect_census", "phase_mapping"]):
    try:
        plot_ablation_bar(world, task_name, seeds=3, ax=axes[i])
    except Exception as e:
        axes[i].text(0.5, 0.5, f"Error: {e}", transform=axes[i].transAxes,
                     ha="center", va="center")
        axes[i].set_title(task_name)

fig.suptitle("Ablation Study: Component Contributions", fontsize=16, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "ablation_bar.pdf")
fig.savefig(FIGURES_DIR / "ablation_bar.png")
plt.show()

## 3. Decision Trace

Visualize where the agent looked and what planner modes it used.
Shows trajectory on top of overview + mode transitions timeline.

In [ ]:
def plot_decision_trace(
    world_path: str,
    agent_name: str = "stem_agent",
    task_name: str = "defect_census",
    budget: float = 5000.0,
    seed: int = 42,
):
    """Run one episode and plot the decision trace."""
    from stem_gym.envs.stem_env import STEMEnv
    from stem_gym.envs.world import World
    from stem_gym.runner import run_episode
    from experiments.run_benchmark import make_agent, make_task

    task = make_task(task_name)
    env = STEMEnv(world_path, task=task, total_budget=budget, max_steps=200)
    agent = make_agent(agent_name, seed=seed)
    result = run_episode(env, agent, seed=seed, score_every_n_acquires=3)

    # Get world overview
    world = World(world_path)
    overview = world.get_overview()
    pixel_size = world.pixel_size_nm
    world.close()
    env.close()

    # Extract trajectory positions
    positions = [entry["position"] for entry in result.episode_log if entry.get("position") is not None]

    # Extract transition log if available
    findings = result.agent_findings

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: Trajectory on overview
    ax = axes[0]
    ax.imshow(overview, cmap="gray", origin="lower")
    if positions:
        xs = [p[0] / pixel_size for p in positions]
        ys = [p[1] / pixel_size for p in positions]
        # Color by step (early=blue, late=red)
        colors = plt.cm.coolwarm(np.linspace(0, 1, len(xs)))
        for i in range(len(xs) - 1):
            ax.plot([xs[i], xs[i+1]], [ys[i], ys[i+1]],
                    color=colors[i], alpha=0.7, lw=1.5)
        ax.plot(xs[0], ys[0], "go", markersize=10, label="Start", zorder=5)
        ax.plot(xs[-1], ys[-1], "rs", markersize=10, label="End", zorder=5)
        ax.legend(loc="upper right")
    ax.set_title("Agent Trajectory")

    # Panel 2: DEC curve for this episode
    ax = axes[1]
    ax.plot(result.dec.dose_fractions, result.dec.scores, "b-", lw=2)
    ax.fill_between(result.dec.dose_fractions, 0, result.dec.scores, alpha=0.2, color="blue")
    ax.set_xlabel("Dose Fraction")
    ax.set_ylabel("Task Score")
    ax.set_title(f"DEC Curve (AUC={result.dec.auc:.3f})")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)

    # Panel 3: Score and dose over time
    ax = axes[2]
    steps = [e["step"] for e in result.episode_log]
    doses = [e["dose_used"] / budget for e in result.episode_log]
    scores = []
    last_score = 0.0
    for e in result.episode_log:
        if e.get("task_score") is not None:
            last_score = e["task_score"]
        scores.append(last_score)

    ax.plot(steps, doses, "r-", label="Dose fraction", alpha=0.7)
    ax.plot(steps, scores, "b-", label="Task score", lw=2)
    ax.set_xlabel("Step")
    ax.set_ylabel("Value")
    ax.set_title("Episode Timeline")
    ax.legend()
    ax.grid(True, alpha=0.3)

    style = get_style(agent_name)
    fig.suptitle(f"Decision Trace: {style['label']} on {task_name}", fontsize=14, y=1.02)
    plt.tight_layout()
    return fig


try:
    fig = plot_decision_trace(world, "stem_agent", "defect_census", budget=5000.0)
    fig.savefig(FIGURES_DIR / "decision_trace.pdf")
    fig.savefig(FIGURES_DIR / "decision_trace.png")
    plt.show()
except Exception as e:
    print(f"Error: {e}")

## 4. Simulation vs Replay Correlation

Compare agent rankings on simulated data vs. experimental replay data.
A strong correlation validates that simulation rankings generalize.

In [ ]:
def plot_sim_vs_replay(
    sim_path: str,
    replay_path: str,
    task_name: str = "defect_census",
    budget: float = 5000.0,
    seeds: int = 3,
):
    """Compare agent DEC-AUC on sim vs replay data."""
    from stem_gym.envs.stem_env import STEMEnv
    from stem_gym.runner import run_benchmark
    from experiments.run_benchmark import make_agent, make_task

    agents = ["random", "raster", "gp_ucb", "stem_agent"]
    task = make_task(task_name)

    sim_aucs = []
    replay_aucs = []
    labels = []

    for agent_name in agents:
        style = get_style(agent_name)
        labels.append(style["label"])

        # Sim
        env = STEMEnv(sim_path, task=task, total_budget=budget, max_steps=200)
        agent = make_agent(agent_name)
        bench = run_benchmark(env, agent, agent_name, n_seeds=seeds, score_every_n_acquires=3)
        sim_aucs.append(bench.mean_dec_auc)
        env.close()

        # Replay
        env = STEMEnv(replay_path, task=task, total_budget=budget, max_steps=200)
        agent = make_agent(agent_name)
        bench = run_benchmark(env, agent, agent_name, n_seeds=seeds, score_every_n_acquires=3)
        replay_aucs.append(bench.mean_dec_auc)
        env.close()

    fig, ax = plt.subplots(figsize=(6, 6))

    for i, (sx, ry, label) in enumerate(zip(sim_aucs, replay_aucs, labels)):
        style = get_style(agents[i])
        ax.scatter(sx, ry, s=120, c=style["color"], marker=style["marker"],
                   label=label, zorder=5, edgecolors="black")

    # Perfect correlation line
    lim = max(max(sim_aucs), max(replay_aucs)) * 1.1
    ax.plot([0, lim], [0, lim], "k--", alpha=0.3, label="y=x")

    # Spearman rank correlation
    from scipy.stats import spearmanr
    rho, pval = spearmanr(sim_aucs, replay_aucs)
    ax.text(0.05, 0.95, f"Spearman rho={rho:.2f}\np={pval:.3f}",
            transform=ax.transAxes, va="top", fontsize=11,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

    ax.set_xlabel("DEC-AUC (Simulation)")
    ax.set_ylabel("DEC-AUC (Replay)")
    ax.set_title(f"Sim vs Replay — {TASK_LABELS.get(task_name, task_name)}")
    ax.legend(loc="lower right")
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    return fig


sim = str(Path("..") / "data" / "simulated" / "test_world.h5")
replay = str(Path("..") / "data" / "replay" / "replay_world.h5")

try:
    fig = plot_sim_vs_replay(sim, replay, seeds=3)
    fig.savefig(FIGURES_DIR / "sim_vs_replay.pdf")
    fig.savefig(FIGURES_DIR / "sim_vs_replay.png")
    plt.show()
except Exception as e:
    print(f"Error: {e}")

## 5. Summary Table

Generate a formatted results table for the paper.

In [ ]:
def generate_summary_table(
    world_path: str,
    task_name: str = "defect_census",
    budget: float = 5000.0,
    seeds: int = 3,
):
    """Generate summary table of all agents."""
    from stem_gym.envs.stem_env import STEMEnv
    from stem_gym.runner import run_benchmark
    from experiments.run_benchmark import make_agent, make_task

    agents = ["random", "raster", "gp_ucb", "stem_agent",
              "no_planner", "no_uncertainty"]
    task = make_task(task_name)

    rows = []
    for agent_name in agents:
        env = STEMEnv(world_path, task=task, total_budget=budget, max_steps=200)
        agent = make_agent(agent_name)
        bench = run_benchmark(env, agent, agent_name, n_seeds=seeds, score_every_n_acquires=3)
        env.close()

        style = get_style(agent_name)
        rows.append({
            "Agent": style["label"],
            "DEC-AUC": f"{bench.mean_dec_auc:.4f} ({bench.std_dec_auc:.4f})",
            "Final Score": f"{bench.mean_final_score:.4f}",
            "Tiles": f"{bench.mean_tiles:.1f}",
            "Dose Used": f"{bench.mean_dose_used:.0f}",
        })

    # Print as formatted table
    headers = list(rows[0].keys())
    col_widths = {h: max(len(h), max(len(r[h]) for r in rows)) for h in headers}

    # Header
    header_str = " | ".join(h.ljust(col_widths[h]) for h in headers)
    sep_str = "-+-".join("-" * col_widths[h] for h in headers)
    print(header_str)
    print(sep_str)

    # Rows
    for row in rows:
        print(" | ".join(row[h].ljust(col_widths[h]) for h in headers))

    # LaTeX version
    print("\n\n% LaTeX table:")
    print("\\begin{tabular}{l" + "r" * (len(headers) - 1) + "}")
    print("\\toprule")
    print(" & ".join(headers) + " \\\\")
    print("\\midrule")
    for row in rows:
        print(" & ".join(row[h] for h in headers) + " \\\\")
    print("\\bottomrule")
    print("\\end{tabular}")

    return rows


try:
    print(f"Task: Defect Census on test_world\n")
    rows = generate_summary_table(world, "defect_census", seeds=3)
except Exception as e:
    print(f"Error: {e}")

## 6. Load Pre-computed Results

If you've already run the full benchmark via CLI, load and visualize those results.

In [ ]:
def plot_from_saved_results(results_file: str):
    """Plot DEC-AUC bar chart from pre-saved JSON results."""
    data = load_results(results_file)
    if data is None:
        return

    results = data["results"]

    # Group by task
    tasks_seen = sorted(set(r["task"] for r in results))
    n_tasks = len(tasks_seen)

    fig, axes = plt.subplots(1, max(n_tasks, 1), figsize=(7 * n_tasks, 5))
    if n_tasks == 1:
        axes = [axes]

    for idx, task_name in enumerate(tasks_seen):
        ax = axes[idx]
        task_results = [r for r in results if r["task"] == task_name]

        # Average across worlds if multiple
        agent_scores = {}
        for r in task_results:
            name = r["agent"]
            agent_scores.setdefault(name, []).append(r["mean_dec_auc"])

        names = list(agent_scores.keys())
        means = [np.mean(v) for v in agent_scores.values()]
        stds = [np.std(v) for v in agent_scores.values()]

        # Sort by mean DEC-AUC
        order = np.argsort(means)[::-1]
        names = [names[i] for i in order]
        means = [means[i] for i in order]
        stds = [stds[i] for i in order]
        colors = [get_style(n)["color"] for n in names]
        display_names = [get_style(n)["label"] for n in names]

        x = np.arange(len(names))
        bars = ax.bar(x, means, yerr=stds, capsize=4, color=colors,
                      edgecolor="black", alpha=0.85)

        for bar, m in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f"{m:.3f}", ha="center", va="bottom", fontsize=9)

        ax.set_xticks(x)
        ax.set_xticklabels(display_names, rotation=20, ha="right")
        ax.set_ylabel("DEC-AUC")
        ax.set_title(TASK_LABELS.get(task_name, task_name))
        ax.grid(axis="y", alpha=0.3)

    fig.suptitle(f"Results: {data['config']['name']}", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


# Try loading pre-computed results
for name in ["full_benchmark", "ablation", "smoke"]:
    data = load_results(name)
    if data is not None:
        print(f"\n--- {name} ---")
        plot_from_saved_results(name)